# How to read an Avro file with Python

Load Avro into pandas, read the embedded schema, select fields with a reader schema, filter records while streaming, and convert Avro to Parquet for repeated queries.

This notebook accompanies [How to read an Avro file with Python](https://docs.opteryx.app/docs/guides/read-an-avro-file-with-python) on docs.opteryx.app. It builds its own sample data, so it runs top to bottom with nothing to download.

In [ ]:
%pip install -q pandas pyarrow fastavro polars opteryx-core

## Make some sample data

200,000 synthetic orders, sorted by `ordered_at`. Every example below reads this file.

In [ ]:
import datetime
import random

import pyarrow

random.seed(42)
N = 200_000
start = datetime.datetime(2024, 1, 1)

orders = pyarrow.table({
    "order_id": list(range(1, N + 1)),
    "customer_id": [random.randint(1, 20_000) for _ in range(N)],
    "country": [random.choice(["GB", "US", "DE", "FR", "JP", "AU", "CA", "NL"]) for _ in range(N)],
    "status": random.choices(["delivered", "shipped", "pending", "cancelled", "returned"], weights=[60, 15, 10, 10, 5], k=N),
    "amount": [round(random.lognormvariate(3.5, 0.8), 2) for _ in range(N)],
    "ordered_at": sorted(start + datetime.timedelta(seconds=random.randint(0, 365 * 86400)) for _ in range(N)),
})
orders.num_rows

In [ ]:
import fastavro

schema = fastavro.parse_schema({
    "type": "record",
    "name": "Order",
    "fields": [
        {"name": "order_id", "type": "long"},
        {"name": "customer_id", "type": "long"},
        {"name": "country", "type": "string"},
        {"name": "status", "type": "string"},
        {"name": "amount", "type": "double"},
        {"name": "ordered_at", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    ],
})

with open("orders.avro", "wb") as f:
    fastavro.writer(f, schema, orders.to_pylist(), codec="deflate")

## The short answer: fastavro and pandas

pandas has no Avro reader, so read records with fastavro and build a DataFrame from them.

In [ ]:
import pandas as pd

with open("orders.avro", "rb") as f:
    df = pd.DataFrame.from_records(fastavro.reader(f))

df.dtypes

In [ ]:
df.head()

## Read the schema first

The header comes before any records, so the schema and codec are available immediately.

In [ ]:
with open("orders.avro", "rb") as f:
    reader = fastavro.reader(f)
    print(reader.codec)
    print(reader.writer_schema)

## Read only some fields

A reader schema is the schema you want; fields you leave out are dropped.

In [ ]:
reader_schema = {
    "type": "record",
    "name": "Order",
    "fields": [
        {"name": "order_id", "type": "long"},
        {"name": "amount", "type": "double"},
    ],
}

with open("orders.avro", "rb") as f:
    df = pd.DataFrame.from_records(fastavro.reader(f, reader_schema=reader_schema))

df.head()

## Filter while reading

Filter in a generator so pandas only sees the rows you keep. Every record is still decoded.

In [ ]:
def matching_orders(path):
    with open(path, "rb") as f:
        for record in fastavro.reader(f):
            if record["country"] == "GB" and record["amount"] > 100:
                yield {"order_id": record["order_id"], "amount": record["amount"]}

df = pd.DataFrame(matching_orders("orders.avro"))
len(df)

In [ ]:
# or a block at a time
parts = []
with open("orders.avro", "rb") as f:
    for block in fastavro.block_reader(f):
        chunk = pd.DataFrame.from_records(list(block))
        parts.append(chunk[chunk["status"] == "returned"])

len(pd.concat(parts, ignore_index=True))

## Polars

Polars reads Avro natively with a column list. Its Avro support is marked unstable.

In [ ]:
import polars as pl

df = (
    pl.read_avro("orders.avro", columns=["order_id", "country", "amount"])
      .filter((pl.col("country") == "GB") & (pl.col("amount") > 100))
      .to_pandas()
)
len(df)

## Rugo (experimental)

Rugo's native Avro reader is in development and not in a released `rugo` wheel yet, so this cell skips itself if it isn't available.

In [ ]:
try:
    import pyarrow
    from rugo.avro import read_avro, read_metadata
except ImportError:
    print("rugo.avro is not in the installed rugo release yet")
else:
    print(read_metadata("orders.avro"))
    with read_avro("orders.avro", columns=["order_id", "country", "amount"]) as reader:
        df = pyarrow.concat_tables(morsel.to_arrow() for morsel in reader).to_pandas()
    print(len(df))

## Querying repeatedly: convert to Parquet

Decode once, write Parquet, and every later read can skip columns and row groups.

In [ ]:
import pyarrow
import pyarrow.parquet as pq

with open("orders.avro", "rb") as f:
    table = pyarrow.Table.from_pylist(list(fastavro.reader(f)))

pq.write_table(table, "orders.parquet")
table.schema

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT country, COUNT(*) AS orders, ROUND(AVG(amount), 2) AS avg_amount
      FROM READ_PARQUET('orders.parquet')
     WHERE status = 'delivered'
     GROUP BY country
     ORDER BY orders DESC
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()